# Tarea 21 · Partición de datos y codificación del objetivo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/21_particion_datos.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel.

Separamos los predictores del objetivo `Churn`, codificamos las bajas y reservamos un 20 % para evaluación. Entrada: `data/processed/telco_clean.csv`. Salidas: cuatro CSV sin transformar en `data/processed/particion/`, con el índice original para conservar la correspondencia entre clientes y etiquetas.

## Cómo ejecutarlo

- **Local:** instalar `requirements.txt`, abrir el notebook dentro del repositorio y ejecutar todas las celdas en orden.
- **Colab:** usar el botón cuando la rama está publicada, o subir este notebook; el dataset se obtiene desde `main` de GitHub. Las salidas quedan en el panel **Archivos**.


## 1. Carga del dataset limpio

Buscamos la raíz del repositorio desde el directorio actual o sus padres. En Colab usamos el CSV publicado. Validamos el objetivo antes de codificarlo.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

if EN_COLAB:
    RAIZ = Path.cwd()
    fuente = 'https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/data/processed/telco_clean.csv'
else:
    actual = Path.cwd().resolve()
    RAIZ = next((ruta for ruta in (actual, *actual.parents)
                 if (ruta / 'README.md').is_file()), None)
    if RAIZ is None:
        raise FileNotFoundError('Abrí Jupyter dentro del repositorio del TPI.')
    fuente = RAIZ / 'data' / 'processed' / 'telco_clean.csv'

df = pd.read_csv(fuente)
assert df.index.is_unique
assert not df.isna().any().any(), 'La entrada debe ser el dataset limpio, sin nulos.'
assert set(df['Churn'].unique()) == {'No', 'Yes'}
assert 'customerID' not in df.columns
print(f'Dataset limpio: {df.shape[0]:,} clientes y {df.shape[1]} columnas.')

Dataset limpio: 7,043 clientes y 22 columnas.


## 2. Predictores y objetivo

`X` contiene todas las columnas excepto `Churn`. `LabelEncoder` se aplica exclusivamente al objetivo: `No` corresponde a 0 y `Yes` a 1. Esta equivalencia fija no aprende estadísticas de los predictores. Las categorías de entrada y sus escalas se ajustarán después de partir, usando solo entrenamiento.

In [2]:
X = df.drop(columns='Churn').copy()
label_encoder = LabelEncoder()
y = pd.Series(label_encoder.fit_transform(df['Churn']), index=df.index, name='Churn')
mapeo = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
assert mapeo == {'No': 0, 'Yes': 1}
assert 'Churn' not in X.columns
assert X.index.equals(y.index)
print(f'Codificación: {mapeo}')
print(f'Predictores por cliente: {X.shape[1]}')

Codificación: {'No': np.int64(0), 'Yes': np.int64(1)}
Predictores por cliente: 21


## 3. Partición estratificada 80/20

Reservamos test antes de codificar o escalar los predictores. La estratificación conserva la proporción de bajas y `random_state=42` permite repetir la misma partición. Los índices identifican las filas del CSV de entrada, no son una variable predictora.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

resumen = pd.DataFrame({
    nombre: {'clientes': len(objetivo), 'bajas': int(objetivo.sum()),
             'tasa_churn_pct': objetivo.mean() * 100}
    for nombre, objetivo in [('Total', y), ('Train', y_train), ('Test', y_test)]
}).T
resumen[['clientes', 'bajas']] = resumen[['clientes', 'bajas']].astype(int)
display(resumen.round(4))

,clientes,bajas,tasa_churn_pct
Total,7043,1869,26.5370
Train,5634,1495,26.5353
Test,1409,374,26.5436


La tabla permite comparar la tasa de bajas de cada conjunto con la cartera completa. La división 80/20 se redondea a clientes enteros; las diferencias mínimas en las tasas son esperables. Mantener esa composición evita evaluar el modelo con una proporción de bajas artificialmente distinta.

## 4. Verificaciones de la partición

Comprobamos que cada cliente aparezca una sola vez, que las etiquetas sigan alineadas y que la diferencia de tasas sea menor a 0,1 puntos porcentuales. También verificamos que los predictores conserven sus valores originales: todavía no se ajustó ningún codificador ni escalador sobre ellos.

In [4]:
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(df)
assert set(X_train.index) | set(X_test.index) == set(df.index)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert abs(y_train.mean() - y_test.mean()) < 0.001
pd.testing.assert_frame_equal(X_train, X.loc[X_train.index])
pd.testing.assert_frame_equal(X_test, X.loc[X_test.index])
pd.testing.assert_series_equal(y_train, y.loc[y_train.index])
pd.testing.assert_series_equal(y_test, y.loc[y_test.index])
print('Verificaciones correctas: sin solapamientos, sin pérdida de filas y etiquetas alineadas.')
print(f'Diferencia de churn train/test: {abs(y_train.mean() - y_test.mean()) * 100:.4f} puntos porcentuales.')

Verificaciones correctas: sin solapamientos, sin pérdida de filas y etiquetas alineadas.
Diferencia de churn train/test: 0.0083 puntos porcentuales.


## 5. Exportación para las tareas 22 y 23

Guardamos la partición sin transformar en una subcarpeta propia. Los archivos transformados que debe producir la tarea 23 seguirán usando las rutas `data/processed/X_train.csv`, etc. Para leer esta partición, usar `pd.read_csv(ruta, index_col='fila_origen')`; para `y`, seleccionar después la columna `Churn`. Nunca incluir `fila_origen` como predictor.

Las tareas siguientes deben hacer `fit` de codificadores y escaladores únicamente con `X_train` y usar `transform` sobre `X_test`. El conjunto de prueba se reserva para la evaluación final.

In [5]:
carpeta_salida = RAIZ / 'data' / 'processed' / 'particion'
carpeta_salida.mkdir(parents=True, exist_ok=True)
for nombre, datos in [('X_train', X_train), ('X_test', X_test),
                      ('y_train', y_train), ('y_test', y_test)]:
    ruta = carpeta_salida / f'{nombre}.csv'
    datos.to_csv(ruta, index=True, index_label='fila_origen')
    recuperado = pd.read_csv(ruta, index_col='fila_origen')
    esperado = datos.to_frame() if isinstance(datos, pd.Series) else datos
    pd.testing.assert_frame_equal(recuperado.rename_axis(None), esperado)
print('Cuatro archivos exportados y verificados en:', carpeta_salida)

Cuatro archivos exportados y verificados en: D:\src\tpi-analisis-de-datos\data\processed\particion


La partición queda lista para preprocesamiento. El churn conserva su representación `Yes/No` en el dataset limpio; la codificación 0/1 se utiliza en estas etiquetas de modelado. Las columnas calculadas se conservan como predictores disponibles; la selección de variables y las transformaciones deberán justificarse en las etapas siguientes.